# Replacement-Dynamics Adoption Model

This notebook builds a vehicle adoption model that follows the logic discussed in the meeting.

## Core idea

We separate the problem into two pieces:

1. **Sales share can change quickly**
   - The mix of newly entering vehicles can change a lot over time, especially for EVs.
   - We model this with a historical **adoption rate** computed from the yearly sales share by category.

2. **Fleet composition changes more slowly**
   - The total fleet does not explode exponentially because vehicles are replaced gradually.
   - We keep the fleet size close to a stable long-run path using the historical average **net annual fleet change**.

## Modeling assumptions

For each future year:

- We estimate the **sales share** of each vehicle type using historical adoption rates.
- We estimate how many **new vehicles enter** the fleet using the historical average yearly entries.
- We estimate the **total fleet net change** using the historical average yearly fleet change.
- From that, we infer the number of **vehicles leaving the fleet**:
  - `exits = entries - net_change`
- We add new vehicles according to the forecasted sales share.
- We remove vehicles proportionally to the current fleet composition.

This means:

- sales share can evolve fast,
- but fleet share evolves smoothly,
- because replacement takes time.

## Important note on infinity values

When a historical share is zero, the usual growth-rate formula can produce `inf`.

In this notebook, those values are replaced by `0` before averaging, which avoids the previous instability.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 5)

PROJECT_DIR = Path('/Users/natomanzolli/Documents/GitHub/MATSim-agent-vehicle-assignment/adoption prediction model')
CACHE_DIR = PROJECT_DIR / '.cache'

FLEET_COUNTS_CACHE = CACHE_DIR / 'saq_vehicle_counts.pkl'
ENTRY_FULL_CACHE = CACHE_DIR / 'saq_entry_full.pkl'
ENTRY_COUNTS_CACHE = CACHE_DIR / 'saq_entry_counts.pkl'
FLAG_COMPARISON_CACHE = CACHE_DIR / 'saq_entry_flag_comparison.pkl'

fleet_counts = pd.read_pickle(FLEET_COUNTS_CACHE)
entry_full = pd.read_pickle(ENTRY_FULL_CACHE)
entry_counts = pd.read_pickle(ENTRY_COUNTS_CACHE)
flag_comparison = pd.read_pickle(FLAG_COMPARISON_CACHE)

fleet_counts.head()

## Settings

You can change these values if you want to test alternatives.

In [ ]:
ENTRY_FLAG = 'Entrant'  # alternative: 'Neuf'
FORECAST_END_YEAR = 2035
USE_RECENT_YEARS_ONLY = False
RECENT_YEAR_START = 2016

# Optional guardrail for extreme adoption rates.
# Example: 1.0 means max +100% and min -100% annual share growth before averaging.
CLIP_ADOPTION_RATE = None  # set to a number like 1.0 if needed

ENTRY_FLAG, FORECAST_END_YEAR, USE_RECENT_YEARS_ONLY, CLIP_ADOPTION_RATE

## Step 1: Historical Fleet Counts

We start from the observed fleet by year and by vehicle type.

In [ ]:
fleet_yearly = (
    fleet_counts.groupby(['AnneeSAAQ', 'vehicle_type'], as_index=False)['count']
    .sum()
)

fleet_pivot = (
    fleet_yearly.pivot(index='AnneeSAAQ', columns='vehicle_type', values='count')
    .fillna(0)
    .sort_index()
)
fleet_pivot['total_vehicles'] = fleet_pivot.sum(axis=1)
vehicle_cols = [c for c in fleet_pivot.columns if c != 'total_vehicles']

fleet_pivot

## Step 2: Historical Sales Share (Entries)

We compute the yearly mix of newly entering vehicles. This is the part of the system that can evolve rapidly.

In [ ]:
entry_by_year_type = (
    entry_counts.groupby(['AnneeSAAQ', 'vehicle_type'], as_index=False)[ENTRY_FLAG]
    .sum()
    .rename(columns={ENTRY_FLAG: 'entry_count'})
)
entry_by_year_type['entry_share'] = entry_by_year_type.groupby('AnneeSAAQ')['entry_count'].transform(lambda s: s / s.sum())

entry_share_pivot = (
    entry_by_year_type.pivot(index='AnneeSAAQ', columns='vehicle_type', values='entry_share')
    .fillna(0)
    .sort_index()
)
entry_share_pivot

## Step 3: Historical Adoption Rates of Sales Share

For each category and year, we compute:

`adoption_rate_t = (share_t - share_(t-1)) / share_(t-1)`

If the previous share is zero, this creates `inf`. We replace `inf`, `-inf`, and `NaN` with `0`.

This is exactly the guardrail requested in the discussion.

In [ ]:
adoption_rate = entry_share_pivot.pct_change()
adoption_rate = adoption_rate.replace([np.inf, -np.inf], 0).fillna(0)

if CLIP_ADOPTION_RATE is not None:
    adoption_rate = adoption_rate.clip(lower=-CLIP_ADOPTION_RATE, upper=CLIP_ADOPTION_RATE)

adoption_rate

In [ ]:
adoption_source = adoption_rate.copy()
if USE_RECENT_YEARS_ONLY:
    adoption_source = adoption_source[adoption_source.index >= RECENT_YEAR_START]

avg_adoption_rate = adoption_source.mean()
avg_adoption_summary = pd.DataFrame({
    'vehicle_type': avg_adoption_rate.index,
    'avg_adoption_rate': avg_adoption_rate.values,
    'avg_adoption_rate_pct': avg_adoption_rate.values * 100,
}).sort_values('avg_adoption_rate', ascending=False)
avg_adoption_summary

## Step 4: Historical Fleet Net Change

The total fleet is assumed to remain relatively stable. Instead of exponential growth, we estimate a smooth yearly net change directly from the observed fleet size:

`net_change_t = total_fleet_t - total_fleet_(t-1)`

Then we use the **average yearly net change** for the future.

In [ ]:
fleet_change = fleet_pivot[['total_vehicles']].diff().rename(columns={'total_vehicles': 'fleet_net_change'})
fleet_change['fleet_growth_rate'] = fleet_pivot['total_vehicles'].pct_change()
fleet_change

In [ ]:
fleet_change_source = fleet_change.dropna().copy()
if USE_RECENT_YEARS_ONLY:
    fleet_change_source = fleet_change_source[fleet_change_source.index >= RECENT_YEAR_START]

avg_net_change = fleet_change_source['fleet_net_change'].mean()
avg_growth_rate = fleet_change_source['fleet_growth_rate'].mean()

pd.DataFrame({
    'metric': ['avg_net_change_vehicles', 'avg_growth_rate'],
    'value': [avg_net_change, avg_growth_rate],
})

## Step 5: Historical Entries per Year

We also need the absolute number of new vehicles entering the fleet each year, because market share changes through those additions.

In [ ]:
entries_total = entry_full.groupby('AnneeSAAQ', as_index=False)[ENTRY_FLAG].sum().rename(columns={ENTRY_FLAG: 'entries_total'})
entries_total

In [ ]:
entries_source = entries_total.copy()
if USE_RECENT_YEARS_ONLY:
    entries_source = entries_source[entries_source['AnneeSAAQ'] >= RECENT_YEAR_START].copy()

avg_entries_per_year = entries_source['entries_total'].mean()
avg_entries_per_year

## Step 6: Future Sales Share Projection

We start from the last observed sales share and update it with the average historical adoption rate:

`share_(t+1) = share_t * (1 + avg_adoption_rate)`

Then we:

- clip negative values to `0`
- renormalize so the shares sum to `1`

This allows EV sales share to change faster than fleet share, while still keeping a valid probability vector each year.

In [ ]:
last_observed_year = int(fleet_pivot.index.max())
future_years = list(range(last_observed_year + 1, FORECAST_END_YEAR + 1))

sales_share_future = entry_share_pivot.copy()
current_sales_share = sales_share_future.loc[last_observed_year, vehicle_cols].astype(float).copy()

for year in future_years:
    next_share = current_sales_share * (1 + avg_adoption_rate.reindex(vehicle_cols).fillna(0))
    next_share = next_share.clip(lower=0)
    if next_share.sum() > 0:
        next_share = next_share / next_share.sum()
    else:
        next_share = current_sales_share.copy()

    sales_share_future.loc[year, vehicle_cols] = next_share
    current_sales_share = next_share.copy()

sales_share_future = sales_share_future.sort_index()
sales_share_future.tail(15)

## Step 7: Replacement-Dynamics Fleet Forecast

For each future year:

1. `entries = average yearly entries`
2. `net_change = average yearly fleet net change`
3. `exits = entries - net_change`
4. add entries using the forecasted **sales share**
5. remove exits using the **current fleet share**

Using the current fleet share for removals is a replacement assumption:

- the fleet turns over slowly,
- so the stock composition changes more smoothly than sales.


In [ ]:
projected_counts = fleet_pivot.copy()
current_counts = projected_counts.loc[last_observed_year, vehicle_cols].astype(float).copy()

projection_rows = []

for year in future_years:
    entries = float(avg_entries_per_year)
    exits = max(entries - avg_net_change, 0.0)

    current_fleet_share = current_counts / current_counts.sum()
    forecast_sales_share = sales_share_future.loc[year, vehicle_cols].astype(float)

    additions = entries * forecast_sales_share
    removals = exits * current_fleet_share

    next_counts = (current_counts + additions - removals).clip(lower=0)
    next_total = next_counts.sum()

    projected_counts.loc[year, vehicle_cols] = next_counts
    projected_counts.loc[year, 'total_vehicles'] = next_total

    projection_rows.append({
        'year': year,
        'entries': entries,
        'exits': exits,
        'net_change': entries - exits,
        'projected_total_vehicles': next_total,
    })

    current_counts = next_counts.copy()

projection_summary = pd.DataFrame(projection_rows)
projected_market_share = projected_counts[vehicle_cols].div(projected_counts[vehicle_cols].sum(axis=1), axis=0)
projection_summary

## Main Tables

In [ ]:
avg_adoption_summary

In [ ]:
projection_summary

In [ ]:
projected_counts.tail(15)

## Plot 1: Historical Fleet Size and Smoothed Future Fleet

This should show that total vehicles evolve smoothly, without exponential blow-up.

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(projected_counts.index, projected_counts['total_vehicles'], linewidth=2.4, color='#1d3557', label='Projected total fleet')
plt.scatter(fleet_pivot.index, fleet_pivot['total_vehicles'], color='#e63946', s=28, label='Observed total fleet')
plt.axvline(last_observed_year, linestyle='--', color='black', linewidth=1, label='Forecast start')
plt.xlabel('Year')
plt.ylabel('Total vehicles')
plt.title('Total Fleet: Historical and Replacement-Dynamics Forecast')
plt.legend()
plt.tight_layout()
plt.show()

## Plot 2: Historical and Future Sales Share

This shows the part of the model that can change more rapidly.

In [ ]:
plt.figure(figsize=(12, 6))
for vehicle_type in vehicle_cols:
    plt.plot(sales_share_future.index, sales_share_future[vehicle_type] * 100, marker='o', linewidth=2, label=vehicle_type)
plt.axvline(last_observed_year, linestyle='--', color='black', linewidth=1, label='Forecast start')
plt.xlabel('Year')
plt.ylabel('Sales share (%)')
plt.title(f'Sales Share Forecast Using Historical {ENTRY_FLAG} Adoption Rates')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Plot 3: Fleet Market Share

This is the key output. Fleet market share changes more smoothly than sales share because the fleet turns over gradually.

In [ ]:
plt.figure(figsize=(12, 6))
for vehicle_type in vehicle_cols:
    plt.plot(projected_market_share.index, projected_market_share[vehicle_type] * 100, marker='o', linewidth=2, label=vehicle_type)
plt.axvline(last_observed_year, linestyle='--', color='black', linewidth=1, label='Forecast start')
plt.xlabel('Year')
plt.ylabel('Fleet market share (%)')
plt.title('Fleet Market Share Under Replacement Dynamics')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Plot 4: Vehicle Counts by Type

This lets you check whether the category trajectories are behaving reasonably.

In [ ]:
plt.figure(figsize=(12, 6))
for vehicle_type in vehicle_cols:
    plt.plot(projected_counts.index, projected_counts[vehicle_type], marker='o', linewidth=2, label=vehicle_type)
plt.axvline(last_observed_year, linestyle='--', color='black', linewidth=1, label='Forecast start')
plt.xlabel('Year')
plt.ylabel('Vehicle count')
plt.title('Vehicle Counts by Type Under Replacement Dynamics')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Plot 5: Historical Entries and Fleet Net Change

This plot is useful to see the difference between:

- how many vehicles are entering,
- and how much the total fleet is actually changing.

That gap is what creates replacement dynamics.

In [ ]:
compare_df = entries_total.merge(
    fleet_change.reset_index()[['AnneeSAAQ', 'fleet_net_change']],
    on='AnneeSAAQ',
    how='left'
)
compare_long = compare_df.melt(id_vars='AnneeSAAQ', value_vars=['entries_total', 'fleet_net_change'], var_name='series', value_name='value')

plt.figure(figsize=(11, 5))
sns.lineplot(data=compare_long, x='AnneeSAAQ', y='value', hue='series', marker='o')
plt.xlabel('Year')
plt.ylabel('Vehicles')
plt.title(f'Historical {ENTRY_FLAG} Vehicles vs Historical Fleet Net Change')
plt.tight_layout()
plt.show()

## Optional: Save Outputs

In [ ]:
output_dir = PROJECT_DIR / 'validation_outputs' / 'replacement_dynamics_adoption_model'
output_dir.mkdir(parents=True, exist_ok=True)

avg_adoption_summary.to_csv(output_dir / 'average_adoption_rates.csv', index=False)
entries_total.to_csv(output_dir / f'historical_{ENTRY_FLAG.lower()}_totals.csv', index=False)
projection_summary.to_csv(output_dir / 'projection_summary.csv', index=False)
sales_share_future.to_csv(output_dir / 'projected_sales_share.csv')
projected_counts.to_csv(output_dir / 'projected_fleet_counts.csv')
projected_market_share.to_csv(output_dir / 'projected_fleet_market_share.csv')

print(f'Saved outputs to {output_dir}')